# Module 2 - Filtrage collaboratif avec Surprise

## Objectifs

1. Utiliser la librairie **scikit-surprise** pour prototyper rapidement plusieurs algorithmes de CF.
2. Comparer **BaselineOnly**, **KNNWithMeans**, **SVD**, **NMF** sur RMSE / MAE.
3. Comparer les memes modeles sur des **metriques de ranking** (Precision@K, NDCG, HR).
4. Faire un **GridSearch** sur SVD pour trouver les meilleurs hyperparametres.
5. Comprendre pourquoi le **cosinus item-item** bat SVD sur le ranking (insight cle).

## Algorithmes Surprise

| Algo          | Idee                                                          |
|---------------|---------------------------------------------------------------|
| BaselineOnly  | b_ui = mu + b_u + b_i (biais appris par ALS)                  |
| KNNWithMeans  | CF par plus proches voisins avec centrage                     |
| SVD           | Factorisation matricielle avec biais (Funk SVD + regularisee) |
| NMF           | Factorisation non-negative (lee & seung)                      |
| CoClustering  | Co-clustering user/item                                       |


## 1. Setup

In [ ]:
import sys, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)

# Surprise wants to write to ~/.surprise_data but we lack permissions; redirect.
os.environ.setdefault("SURPRISE_DATA_FOLDER", str(Path("..").resolve() / ".surprise_data"))

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.loaders import load_movielens_1m, temporal_split
from src.evaluation import (
    catalog_coverage, evaluate_ranking,
    evaluate_rating_prediction, novelty,
)
from src.models import ItemItemCosine, PopularityBaseline
from src.models.surprise_models import make_surprise_model

print("OK")

## 2. Chargement + split temporel

In [ ]:
DATA_ROOT = ROOT / "data" / "raw" / "ml-1m"
ml = load_movielens_1m(DATA_ROOT)
print(f"Ratings : {ml.n_ratings:,}")

train, test = temporal_split(ml.ratings, test_ratio=0.2)
print(f"Train : {len(train):,}  Test : {len(test):,}")

train_users = set(train["user_id"].unique())
train_items = set(train["movie_id"].unique())
test_eval = test[test["user_id"].isin(train_users) & test["movie_id"].isin(train_items)].copy()

rng = np.random.default_rng(0)
eval_users = list(rng.choice(test_eval["user_id"].unique(), size=1000, replace=False))
test_subset = test_eval[test_eval["user_id"].isin(eval_users)]

## 3. Comparaison des modeles (full data)

In [ ]:
models = {
    "BaselineOnly": make_surprise_model("BaselineOnly"),
    "SVD (50)": make_surprise_model("SVD", n_factors=50, n_epochs=20, random_state=0),
    "NMF (50)": make_surprise_model("NMF", n_factors=50, n_epochs=20, random_state=0),
    "CoClustering": make_surprise_model("CoClustering", n_epochs=20, random_state=0),
}
pop = PopularityBaseline(min_ratings=50, score="count").fit(train)
cos = ItemItemCosine().fit(train)

fit_times = {}
for name, m in models.items():
    t0 = time.time()
    m.fit(train)
    fit_times[name] = time.time() - t0
    print(f"  {name:<18} fit {fit_times[name]:.1f}s")

In [ ]:
# Rating prediction
pairs = test_eval[["user_id", "movie_id"]].drop_duplicates().head(50_000)
pairs = pairs.merge(test_eval[["user_id", "movie_id", "rating"]], on=["user_id", "movie_id"], how="left").dropna()

rows = []
for name, m in models.items():
    preds = m.predict_for_pairs(pairs[["user_id", "movie_id"]])
    met = evaluate_rating_prediction(pairs, preds)
    met["model"] = name; met["fit_s"] = fit_times[name]
    rows.append(met)
rmse_df = pd.DataFrame(rows).set_index("model")[["rmse", "mae", "coverage", "fit_s"]]
display(rmse_df.round(4))

In [ ]:
# Top-K ranking
all_recs = {
    "Popularity": pop.recommend_for_users(eval_users, top_k=50),
    "ItemItemCosine": cos.recommend_for_users(eval_users, top_k=50),
}
for name, m in models.items():
    all_recs[name] = m.recommend_for_users(eval_users, top_k=50)

catalog = set(train_items)
item_pop = train.groupby("movie_id").size().to_dict()

rows = []
for name, recs in all_recs.items():
    r = evaluate_ranking(recs, test_subset, k=10)
    r["model"] = name
    r["catalog_coverage@10"] = catalog_coverage(recs, catalog, k=10)
    r["novelty@10"] = novelty(recs, item_pop, k=10)
    r["fit_s"] = fit_times.get(name, 0.0)
    rows.append(r)

rank_df = pd.DataFrame(rows).set_index("model")
rank_df = rank_df[["precision@10", "recall@10", "ndcg@10", "hit_rate@10", "map@10",
                   "catalog_coverage@10", "novelty@10", "fit_s"]]
display(rank_df.round(4))

## 4. Insight : pourquoi cosine bat SVD sur le ranking ?

Surprise : `predict(u, i)` renvoie la note predite. Pour recommander, on classe tous les items par `predict(u, i)` decroissant. Pour les items non vus, la prediction est dominee par `mu + b_u + b_i` (biais) plus un terme `p_u . q_i` generalise.

Notre cosine baseline (Module 1) calcule directement :

$$
\\mathrm{score}(u, i) = \\sum_{j \\in \\mathrm{seen}(u),\\, r_{uj} > \\bar r_u} \\mathrm{sim}(i, j) \\cdot (r_{uj} - \\bar r_u)
$$

C'est un score **plus agressif** : il amplifie les items tres similaires a ceux que l utilisateur a sur-notes. SVD generalise (facteurs latents compresses) ; cosine memorise les similarites item-item.

Sur RMSE, SVD gagne (0.871 vs 0.955) parce que la **generalisation** aide a predire la note exacte. Sur ranking, le **signal local** de cosine l'emporte quand on a beaucoup de notes.

En production, on combine souvent les deux : un modele de scoring (cosine) + un modele de re-ranking (SVD/LightFM).

## 5. Grid search SVD

Surprise fournit un `GridSearchCV` qui optimise le RMSE par validation croisee.

Note : la CV integree fait du split random, pas temporel. On l utilise ici pour reperer les bonnes zones d hyperparametres, puis on reaffinera avec split temporel pour comparer equitablement.

In [ ]:
from surprise import SVD
from surprise.model_selection import GridSearchCV
from surprise import Dataset, Reader

reader = Reader(rating_scale=(1, 5))
full = Dataset.load_from_df(train[["user_id", "movie_id", "rating"]], reader)

param_grid = {
    "n_factors": [50, 100],
    "n_epochs": [20, 40],
    "lr_all": [0.005, 0.01],
    "reg_all": [0.02, 0.1],
}

gs = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=-1, joblib_verbose=0)
gs.fit(full)

print("Best RMSE:", gs.best_score["rmse"])
print("Best params:", gs.best_params["rmse"])

In [ ]:
# Re-fit best SVD on the temporal train split and compare with default
best_params = gs.best_params["rmse"]
svd_best = make_surprise_model("SVD", random_state=0, **best_params)
svd_best.fit(train)

svd_default = make_surprise_model("SVD", n_factors=50, n_epochs=20, random_state=0)
svd_default.fit(train)

preds_best = svd_best.predict_for_pairs(pairs[["user_id", "movie_id"]])
preds_def = svd_default.predict_for_pairs(pairs[["user_id", "movie_id"]])
m_best = evaluate_rating_prediction(pairs, preds_best)
m_def = evaluate_rating_prediction(pairs, preds_def)
print(f"SVD default : RMSE {m_def['rmse']:.4f} | MAE {m_def['mae']:.4f}")
print(f"SVD best    : RMSE {m_best['rmse']:.4f} | MAE {m_best['mae']:.4f}")

## 6. Conclusions Module 2

- **SVD > cosine > BaselineOnly > NMF** sur RMSE / MAE (prediction de note).
- **Cosine > Popularity > SVD > BaselineOnly > NMF** sur ranking (Precision/NDCG/HR).
- Le GridSearch SVD peut gagner ~0.005-0.02 RMSE selon le grid.
- Le cosine baseline est **incroyablement competitif** sur le ranking grace a son caractere non-parametrique et local.
- NMF est notablement plus mauvais que SVD ici - les ratings MovieLens 1M ne sont pas positifs et creux comme NMF les aime.

**Pour aller plus loin (Module 3) :**

- Implementer **FunkSVD from scratch** (descente de gradient sur la factorisation) pour comprendre les maths.
- Comparer l implementation maison avec celle de Surprise.
- Regularisation L2, learning rate, momentum.